# AllDap 채점 모델 비교 (M2, M3, M4)

설계: `docs/superpowers/specs/2026-10-02-judge-selfhost-design.md`

**실행 방법**
1. 런타임 > 런타임 유형 변경 > **T4 GPU**
2. 왼쪽 열쇠 아이콘(비밀값)에 `HF_TOKEN` 을 넣는다(Hugging Face 토큰, 읽기 권한). 노트북 접근을 허용한다.
3. Google Drive 의 `MyDrive/alldap_judge5/` 폴더에 `cases.jsonl` 을 올린다.
4. 런타임 > **모두 실행**. Drive 연결 허용 창이 뜨면 허용한다.
5. 끊기면 같은 순서로 다시 **모두 실행** 한다. 이미 채점한 사례는 건너뛴다.
6. 끝나면 마지막 칸이 `alldap_judge5_results.zip` 을 내려받는다. 중간에 오류로 멈췄으면 마지막 칸만 따로 실행해 그때까지의 결과를 받는다.

모델 하나가 메모리에 안 올라가면 그 모델은 건너뛰고 이유를 `{모델}_skipped.json` 에 남긴다. 다른 모델로 바꿔 끼우지 않는다.

디스크: 원본 가중치가 M3, M4 각 약 30GB, M2 약 48GB 라 합치면 코랩 디스크를 넘는다. 그래서 모델 하나를 끝낼 때마다 그 모델의 내려받은 파일을 지운다(다시 실행하면 다시 받는다).

In [ ]:
!pip install -q -U "transformers>=4.50" accelerate bitsandbytes huggingface_hub

In [ ]:
import hashlib, json, os, platform, shutil, traceback
import torch, transformers, bitsandbytes
from google.colab import drive, userdata, files
from huggingface_hub import hf_hub_download, model_info
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/alldap_judge5'
CASES = f'{DIR}/cases.jsonl'
OUT = f'{DIR}/results'
os.makedirs(OUT, exist_ok=True)
HF_TOKEN = userdata.get('HF_TOKEN')

EXPECTED_SHA256 = ""
MAX_NEW_TOKENS = 512
REPOS = {
    "M3": "Unbabel/M-Prometheus-14B",
    "M4": "Qwen/Qwen2.5-14B-Instruct",
    "M2": "mistralai/Mistral-Small-3.1-24B-Instruct-2503",
}

sha = hashlib.sha256(open(CASES, 'rb').read()).hexdigest()
assert sha == EXPECTED_SHA256, f'시험지 해시가 다릅니다: {sha}. 올린 파일이 맞는지 확인하세요.'
CASE_ROWS = [json.loads(l) for l in open(CASES, encoding='utf-8') if l.strip()]
print(len(CASE_ROWS), '건, 해시 확인 완료')

env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
       'transformers': transformers.__version__, 'bitsandbytes': bitsandbytes.__version__,
       'python': platform.python_version()}
json.dump(env, open(f'{OUT}/env.json', 'w'), ensure_ascii=False, indent=2)
print(env)

In [ ]:
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16)  # T4 는 bf16 을 지원하지 않는다

def load(repo):
    # Mistral 토크나이저는 불러올 때 "정규식이 틀려 토큰화가 잘못된다, fix_mistral_regex=True 를 주라" 고
    # 경고한다(2026-10-02 로컬 확인). 한국어 입력이 잘못 쪼개지면 M2 의 결과가 오염되므로 켠다.
    kw = {'fix_mistral_regex': True} if repo.startswith('mistralai/') else {}
    tok = AutoTokenizer.from_pretrained(repo, token=HF_TOKEN, **kw)
    if tok.chat_template is None:
        # Mistral-Small-3.1 은 대화 틀이 tokenizer_config.json 이 아니라 chat_template.json 에만 있다.
        # 그 파일을 직접 읽어 토크나이저에 단다. 틀의 글자는 저장소의 것 그대로다(우리가 쓰지 않는다).
        # AutoProcessor 로 읽지 않는 이유: 이미지 처리기까지 불러 쓸데없는 의존성(PIL)이 생긴다.
        path = hf_hub_download(repo, 'chat_template.json', token=HF_TOKEN)
        tok.chat_template = json.load(open(path, encoding='utf-8'))['chat_template']
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, quantization_config=QUANT, device_map='auto', token=HF_TOKEN)
    except ValueError as e:
        if 'dispatched on the CPU' in str(e):
            raise   # GPU 메모리가 모자라 일부가 CPU 로 밀린 것이다. 아래 run_model 이 건너뜀으로 처리한다.
        # Mistral-Small-3.1 은 이미지도 받는 모델이라 CausalLM 으로 안 열릴 수 있다. 글만 넣어 쓴다.
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(repo, quantization_config=QUANT, device_map='auto', token=HF_TOKEN)
    model.eval()
    return tok, model

def generate(tok, model, messages):
    # 대화 틀로 글자를 먼저 만들고 토큰으로 바꾼다. apply_chat_template 에 return_tensors 를 주면
    # transformers 판에 따라 텐서 대신 dict 를 돌려줘서, 판과 무관한 이 두 단계로 나눴다.
    # add_special_tokens=False: 틀이 이미 시작 토큰(bos)을 넣으므로 한 번 더 붙이지 않는다.
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    new = out[0, enc['input_ids'].shape[1]:]
    finish = 'length' if new.shape[0] >= MAX_NEW_TOKENS else 'stop'
    return tok.decode(new, skip_special_tokens=True), finish

def done_ids(path):
    if not os.path.exists(path):
        return set()
    return {json.loads(l)['case_id'] for l in open(path, encoding='utf-8') if l.strip()}

def skip_model(key, reason):
    json.dump({'model': key, 'reason': reason}, open(f'{OUT}/{key}_skipped.json', 'w'), ensure_ascii=False)
    print(key, '건너뜀:', reason)

def run_model(key):
    repo = REPOS[key]
    skip = f'{OUT}/{key}_skipped.json'
    paths = [f'{OUT}/{key}_run{r}.jsonl' for r in (1, 2)]
    if all(len(done_ids(p)) == len(CASE_ROWS) for p in paths):
        print(key, '이미 끝남'); return
    try:
        revision = model_info(repo, token=HF_TOKEN).sha
        tok, model = load(repo)
        for run, path in zip((1, 2), paths):
            done = done_ids(path)
            for i, row in enumerate(CASE_ROWS):
                if row['case_id'] in done:
                    continue
                text, finish = generate(tok, model, row['messages'])
                line = {'case_id': row['case_id'], 'model': key, 'repo': repo, 'revision': revision,
                        'run': run, 'output': text, 'finish': finish, 'cases_sha256': sha}
                with open(path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(line, ensure_ascii=False) + '\n')
                print(key, f'run{run}', i + 1, '/', len(CASE_ROWS), finish)
        if os.path.exists(skip):
            os.remove(skip)
    except torch.cuda.OutOfMemoryError as e:
        skip_model(key, f'GPU 메모리 부족: {e}')
    except ValueError as e:
        if 'dispatched on the CPU' not in str(e):
            raise
        skip_model(key, f'GPU 메모리 부족(4비트로도 다 올라가지 않아 일부가 CPU 로 밀림): {e}')

In [ ]:
import gc
for key in ('M3', 'M4', 'M2'):
    try:
        run_model(key)
    except Exception:
        # 메모리 부족 말고 다른 이유로 실패하면 멈춘다. 다른 모델로 넘어가 결과를 섞지 않는다.
        traceback.print_exc()
        raise
    finally:
        # run_model 이 돌아오면 그 안의 지역 변수(model, tok)가 사라진다.
        # 그 뒤에 쓰레기 수거와 GPU 캐시 비우기를 해야 다음 모델이 올라갈 자리가 생긴다.
        gc.collect()
        torch.cuda.empty_cache()
        # 내려받은 가중치를 지운다. 세 모델을 합치면 약 107GB 라 코랩 디스크에 다 남길 수 없다.
        shutil.rmtree(os.path.join(HF_HUB_CACHE, 'models--' + REPOS[key].replace('/', '--')), ignore_errors=True)

In [ ]:
shutil.make_archive('/content/alldap_judge5_results', 'zip', OUT)
files.download('/content/alldap_judge5_results.zip')